# 01 · Extract from the Collibra `Account` global data category

This notebook reads the Collibra business-glossary export of the **`Account`
global data category** and pulls out the three things we need:

1. **Business Terms** – assets of type *Business Term*
2. **Preferred Business-Term Attributes** – the *Preferred Term* attributes of each term
3. **Business-Term Relations** – relations between terms and to the data category

It writes `output/account_terms_extracted.json`, consumed by notebook **02**.

In [1]:
import sys, json
from pathlib import Path

# Make ../src importable so we can reuse the shared paths + namespaces.
SRC = Path.cwd().parent / "src"
sys.path.insert(0, str(SRC))
from common import (COLLIBRA_EXPORT, EXTRACTED_FILE, MAPPING_FILE,
                    HBIM_TTL, HBIM_INFERRED_TTL, FIBO_EXCERPT, OUTPUT_DIR,
                    PREFIXES, bind_all,
                    CAA, FPAS, FSE, REL, CMNS_ID, CMNS_ORG, HBIM)
import pandas as pd
pd.set_option("display.max_colwidth", 60)
print("Project root:", SRC.parent)

Project root: C:\Users\marci\OneDrive\DEV\EDU\AIML\Graph ML\Ontology Engineering\Ontology Repository\FIBO\Ontology Mapping


## Load the Collibra export

In [2]:
with open(COLLIBRA_EXPORT, encoding="utf-8") as fh:
    export = json.load(fh)

category = export["dataCategory"]
print(f"Data category : {category['name']}  (scope={category['scope']})")
print(f"Community      : {export['community']['name']}")
print(f"Domain        : {export['domain']['name']}")
print(f"Assets        : {len(export['assets'])}   Relations: {len(export['relations'])}")

Data category : Account  (scope=Global)
Community      : Enterprise Data Governance
Domain        : Banking Business Glossary
Assets        : 8   Relations: 16


## 1) Business terms

In [3]:
business_terms = [a for a in export["assets"] if a["assetType"] == "Business Term"]
pd.DataFrame([{"id": t["id"], "name": t["name"], "status": t["status"]}
             for t in business_terms])

,id,name,status
0,bt-account,Account,Approved
1,bt-current-account,Current Account,Approved
2,bt-checking-account,Checking Account,Candidate
3,bt-savings-account,Savings Account,Approved
4,bt-deposit-account,Deposit Account,Approved
5,bt-account-holder,Account Holder,Approved
6,bt-account-balance,Account Balance,Approved
7,bt-account-identifier,Account Identifier,Approved


## 2) Preferred business-term attributes

Collibra stores a `Preferred Term` flag plus the preferred label, acronym and
synonyms. We surface those here – they drive which terms become first-class HBIM
classes later (preferred) versus `skos:altLabel` synonyms (non-preferred).

In [4]:
preferred_attributes = {}
for term in business_terms:
    attrs = term.get("attributes", {})
    preferred_attributes[term["id"]] = {
        "name": term["name"],
        "is_preferred": bool(attrs.get("Preferred Term", False)),
        "preferred_label": attrs.get("Preferred Term Label"),
        "acronym": attrs.get("Acronym"),
        "synonyms": attrs.get("Synonym", []),
        "definition": attrs.get("Definition"),
        "status": term.get("status"),
    }

df_pref = pd.DataFrame.from_dict(preferred_attributes, orient="index")
df_pref["synonyms"] = df_pref["synonyms"].apply(lambda s: ", ".join(s) if s else "")
df_pref[["name", "is_preferred", "preferred_label", "acronym", "synonyms"]]

,name,is_preferred,preferred_label,acronym,synonyms
bt-account,Account,True,Account,ACCT,Banking Account
bt-current-account,Current Account,True,Current Account,CA,"Checking Account, Demand Deposit Account"
bt-checking-account,Checking Account,False,None,None,
bt-savings-account,Savings Account,True,Savings Account,SAV,Non-Transaction Deposit Account
bt-deposit-account,Deposit Account,True,Deposit Account,None,
bt-account-holder,Account Holder,True,Account Holder,None,Accountholder
bt-account-balance,Account Balance,True,Account Balance,None,Balance
bt-account-identifier,Account Identifier,True,Account Identifier,None,"IBAN, Account Number"


## 3) Business-term relations

In [5]:
id_to_name = {a["id"]: a["name"] for a in export["assets"]}
id_to_name[category["id"]] = category["name"] + " (Data Category)"

relations = [{
    "source_id": r["source"], "source": id_to_name.get(r["source"], r["source"]),
    "type": r["type"],
    "target_id": r["target"], "target": id_to_name.get(r["target"], r["target"]),
} for r in export["relations"]]

pd.DataFrame(relations)[["source", "type", "target"]]

,source,type,target
0,Account,is classified by,Account (Data Category)
1,Current Account,is classified by,Account (Data Category)
2,Savings Account,is classified by,Account (Data Category)
3,Deposit Account,is classified by,Account (Data Category)
4,Account Holder,is classified by,Account (Data Category)
5,Account Balance,is classified by,Account (Data Category)
6,Account Identifier,is classified by,Account (Data Category)
7,Account,groups,Current Account
8,Account,groups,Savings Account
9,Account,groups,Deposit Account


## Persist the extraction for the next step

In [6]:
extracted = {
    "category": category,
    "business_terms": business_terms,
    "preferred_attributes": preferred_attributes,
    "relations": relations,
}
with open(EXTRACTED_FILE, "w", encoding="utf-8") as fh:
    json.dump(extracted, fh, indent=2)
print("[written]", EXTRACTED_FILE)

[written] C:\Users\marci\OneDrive\DEV\EDU\AIML\Graph ML\Ontology Engineering\Ontology Repository\FIBO\Ontology Mapping\output\account_terms_extracted.json
